# Fraus — Fine-tune do BERTimbau para classificacao de satisfacao

Este notebook treina um classificador de texto de 3 classes (insatisfeito, neutro, satisfeito) a partir do BERTimbau (`neuralmind/bert-base-portuguese-cased`), usando o corpus B2W-Reviews01. Requer GPU — rode no Google Colab com o ambiente de execucao configurado para GPU.

In [ ]:
!pip -q install transformers datasets accelerate evaluate scikit-learn

import torch
assert torch.cuda.is_available(), "Runtime sem GPU. Ambiente de execucao > Alterar tipo > GPU."

from google.colab import drive
drive.mount('/content/drive')

import os
DIR_CHECKPOINT = '/content/drive/MyDrive/fraus/checkpoints'
DIR_SAIDA = '/content/drive/MyDrive/fraus/modelos/bertimbau-satisfacao'
os.makedirs(DIR_CHECKPOINT, exist_ok=True)
os.makedirs(DIR_SAIDA, exist_ok=True)


In [ ]:
import pandas as pd

URL_B2W = "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/main/B2W-Reviews01.csv"
bruto = pd.read_csv(URL_B2W, delimiter=';')

def rotular(nota):
    if nota <= 2:
        return 0   # insatisfeito
    if nota == 3:
        return 1   # neutro
    return 2       # satisfeito

dados = bruto[['review_text', 'overall_rating']].dropna()
dados = dados.rename(columns={'review_text': 'texto'})
dados['rotulo'] = dados['overall_rating'].apply(rotular)

# Balancear: as classes 4-5 dominam o corpus e enviesariam o classificador.
menor_classe = dados['rotulo'].value_counts().min()
dados = (
    dados.groupby('rotulo', group_keys=False)
    .apply(lambda g: g.sample(menor_classe, random_state=42))
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
print(dados['rotulo'].value_counts())


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODELO_BASE = "neuralmind/bert-base-portuguese-cased"
tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)

conjunto = Dataset.from_pandas(dados[['texto', 'rotulo']])
conjunto = conjunto.map(
    lambda lote: tokenizador(lote['texto'], truncation=True, max_length=192),
    batched=True,
)
conjunto = conjunto.rename_column('rotulo', 'labels')
divisao = conjunto.train_test_split(test_size=0.15, seed=42)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

modelo = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, num_labels=3)

def calcular_metricas(predicao):
    preditos = np.argmax(predicao.predictions, axis=1)
    return {
        "acuracia": accuracy_score(predicao.label_ids, preditos),
        "f1_macro": f1_score(predicao.label_ids, preditos, average="macro"),
    }

argumentos = TrainingArguments(
    output_dir=DIR_CHECKPOINT,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True,
    logging_steps=100,
)

treinador = Trainer(
    model=modelo,
    args=argumentos,
    train_dataset=divisao['train'],
    eval_dataset=divisao['test'],
    data_collator=DataCollatorWithPadding(tokenizador),
    compute_metrics=calcular_metricas,
)
treinador.train()


In [ ]:
import json

metricas = treinador.evaluate()
print(metricas)

modelo.save_pretrained(DIR_SAIDA)
tokenizador.save_pretrained(DIR_SAIDA)

with open(f"{DIR_SAIDA}/metricas.json", "w", encoding="utf-8") as arquivo:
    json.dump({
        "acuracia": metricas["eval_acuracia"],
        "f1_macro": metricas["eval_f1_macro"],
        "classes": ["insatisfeito", "neutro", "satisfeito"],
    }, arquivo, ensure_ascii=False, indent=2)

print("Baixe a pasta", DIR_SAIDA, "para modelos/bertimbau-satisfacao/ no repo local.")
